# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, processing, and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and uniquely identified entities through their `@id` fields as per FAIR data principles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print summary
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and entities. All elements are referenced by their `@id` (as required for FAIR linking and interoperability).

Let's list all record sets and their fields by their `@id`.

In [ ]:
# List all available record sets and fields by their @id
for record_set in dataset.record_sets:
    print(f"Record Set Name: {record_set.name}\n @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print('-'*60)

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. We reference record sets and fields by their `@id`. This allows unambiguous selection and access.

In [ ]:
# Extract data from all record sets by their @id
record_sets_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_sets_ids:
    # records() yields dicts where key=field.id
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set {rs_id}.")
    else:
        print(f"No data found for record set {rs_id}.")

# Example: show columns for the main record set (first one listed)
MAIN_RECORD_SET_ID = record_sets_ids[0]
if MAIN_RECORD_SET_ID in dataframes:
    print(f"Columns in record set {MAIN_RECORD_SET_ID}:")
    print(dataframes[MAIN_RECORD_SET_ID].columns.tolist())
    display(dataframes[MAIN_RECORD_SET_ID].head())

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps: filter on a numeric field, normalize, and (optionally) group or summarize by a categorical field. All field accesses use their `@id`.

In [ ]:
# Select primary record set and numeric/categorical fields by their @id

# If there is more than one record set, update accordingly
record_set_id = MAIN_RECORD_SET_ID
df = dataframes[record_set_id].copy()

# Choose a numeric field (by @id) - manually identified from the field list above
# Example assumption: '@id' of the 'Age' field is 'https://api.app.sen.science/frontiers/7862866/age' (update if necessary)
# Let's determine what numeric-like columns are present
numeric_candidates = [col for col in df.columns if df[col].dtype in ('int64', 'float64')]
print("Numeric fields in this record set:", numeric_candidates)
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Chosen numeric field: {numeric_field_id}")
else:
    raise ValueError("No numeric fields to process.")

# Example threshold: select records where this field > 50
threshold = 50
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize this numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a grouping/categorical field by its @id
# Try to identify string/object columns suitable for grouping
categorical_candidates = [col for col in df.columns if df[col].dtype == 'object']
print("Categorical fields in this record set:", categorical_candidates)
if categorical_candidates:
    group_field_id = categorical_candidates[0]
    print(f"Grouping by: {group_field_id}")
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    display(grouped.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field and its relation to the grouping/categorical field. This enables insight into the spread and groupwise differences in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Boxplot by grouping field if available
if categorical_candidates:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=30, ha='right')
    plt.show()

## 6. Conclusion
With `mlcroissant`, we loaded and explored the FAIR^2 record set defined by its Croissant schema. All access used globally unique `@id` identifiers for record sets and fields, ensuring traceability and data integrity as required by FAIR data principles.

Through simple processing and visualizations, we demonstrated: 
- How to extract and filter records by field `@id`.
- How to normalize and analyze numeric values.
- How to summarize and visualize categorical groupings.

This approach ensures reproducible, machine-actionable dataset use—ideal for advanced data science and compliance with open science standards.